In [1]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Load the dataset
df = pd.read_csv('Cleaned_dataset.csv')

# Drop unnecessary columns (keep 'Type' as it is obligatory)
df = df.drop(columns=['Localisation', 'Latitude', 'Longitude', 'Title', 'Tags', 'Tags_length'])

# Separate features and target variable
X = df.drop(columns=['Price in DH'])
y = df['Price in DH']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define numerical and categorical features
numerical_features = ['Area in m²', 'Rooms', 'Bedrooms', 'Bathrooms', 'Floor', 'House Age in ans']
categorical_features = ['Condition', 'Type']  # Include 'Type' as a categorical feature

# Preprocessing for numerical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Handle missing values with median
    ('scaler', StandardScaler())  # Standardize numerical features
])

# Preprocessing for categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Handle missing values with mode
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # One-hot encode categorical features
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Apply preprocessing to the data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# Define the neural network model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),  # Input layer
    Dropout(0.2),  # Dropout to prevent overfitting
    Dense(64, activation='relu'),  # Hidden layer
    Dropout(0.2),  # Dropout to prevent overfitting
    Dense(32, activation='relu'),  # Hidden layer
    Dense(1)  # Output layer for regression (no activation function)
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

# Evaluate the model on the test set
loss = model.evaluate(X_test, y_test)
print(f'Test Loss (MSE): {loss}')

# Make predictions
y_pred = model.predict(X_test)

# Compare some actual vs predicted values
comparison = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred.flatten()})
print(comparison.head())

Epoch 1/50


C:\Users\HP\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 56797796761600.0000 - val_loss: 69725589602304.0000
Epoch 2/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 81931156848640.0000 - val_loss: 69712281075712.0000
Epoch 3/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 58813034004480.0000 - val_loss: 69623168892928.0000
Epoch 4/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 66581388328960.0000 - val_loss: 69348622336000.0000
Epoch 5/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 54258523176960.0000 - val_loss: 68708642848768.0000
Epoch 6/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 68567089283072.0000 - val_loss: 67487966167040.0000
Epoch 7/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 68926822154240.0000 - val_loss: 65552743661568.0000
Epoch 8/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 53952443842560.0000 - val_loss: 62515996262400.0000
Epoch 9/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 53688471126016.0000 - val_loss: 58535505297408.0000
Epoch 10/50
66/66 ━━━

In [3]:
import numpy as np

# Calculate Mean Absolute Percentage Error (MAPE)
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Calculate Custom Accuracy (percentage of predictions within a certain threshold)
def custom_accuracy(y_true, y_pred, threshold=0.1):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    within_threshold = np.abs((y_true - y_pred) / y_true) <= threshold
    return np.mean(within_threshold) * 100

# Make predictions
y_pred = model.predict(X_test).flatten()

# Calculate MAPE
mape = mean_absolute_percentage_error(y_test, y_pred)
print(f'Mean Absolute Percentage Error (MAPE): {mape:.2f}%')

# Calculate Custom Accuracy (e.g., within 10% of actual values)
accuracy = custom_accuracy(y_test, y_pred, threshold=0.1)
print(f'Custom Accuracy (within 10% of actual values): {accuracy:.2f}%')

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Mean Absolute Percentage Error (MAPE): 386.75%
Custom Accuracy (within 10% of actual values): 16.82%


In [6]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import numpy as np

# Load the dataset
df = pd.read_csv('Cleaned_dataset.csv')

# Drop unnecessary columns (keep 'Type' as it is obligatory)
df = df.drop(columns=['Localisation', 'Latitude', 'Longitude', 'Title', 'Tags', 'Tags_length'])

# Separate features and target variable
X = df.drop(columns=['Price in DH'])
y = df['Price in DH']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define numerical and categorical features
numerical_features = ['Area in m²', 'Rooms', 'Bedrooms', 'Bathrooms', 'Floor', 'House Age in ans']
categorical_features = ['Condition', 'Type']  # Include 'Type' as a categorical feature

# Preprocessing for numerical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Handle missing values with median
    ('scaler', StandardScaler())  # Standardize numerical features
])

# Preprocessing for categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Handle missing values with mode
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # One-hot encode categorical features
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Apply preprocessing to the data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# Define the neural network model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),  # Input layer
    Dropout(0.3),  # Dropout to prevent overfitting
    Dense(64, activation='relu'),  # Hidden layer
    Dropout(0.3),  # Dropout to prevent overfitting
    Dense(32, activation='relu'),  # Hidden layer
    Dense(1)  # Output layer for regression (no activation function)
])

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mean_squared_error')

# Train the model
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2)

# Evaluate the model on the test set
loss = model.evaluate(X_test, y_test)
print(f'Test Loss (MSE): {loss}')

# Make predictions
y_pred = model.predict(X_test).flatten()

# Calculate Mean Absolute Percentage Error (MAPE)
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Calculate Custom Accuracy (percentage of predictions within a certain threshold)
def custom_accuracy(y_true, y_pred, threshold=0.1):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    within_threshold = np.abs((y_true - y_pred) / y_true) <= threshold
    return np.mean(within_threshold) * 100

# Calculate MAPE
mape = mean_absolute_percentage_error(y_test, y_pred)
print(f'Mean Absolute Percentage Error (MAPE): {mape:.2f}%')

# Calculate Custom Accuracy (e.g., within 10% of actual values)
accuracy = custom_accuracy(y_test, y_pred, threshold=0.1)
print(f'Custom Accuracy (within 10% of actual values): {accuracy:.2f}%')

C:\Users\HP\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - loss: 63722164846592.0000 - val_loss: 69725807706112.0000
Epoch 2/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 63050392535040.0000 - val_loss: 69716127252480.0000
Epoch 3/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 70992101638144.0000 - val_loss: 69657553797120.0000
Epoch 4/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 56708353228800.0000 - val_loss: 69456835379200.0000
Epoch 5/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 58654824857600.0000 - val_loss: 68964348592128.0000
Epoch 6/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 68718184890368.0000 - val_loss: 67967035375616.0000
Epoch 7/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 67920205971456.0000 - val_loss: 66354937856000.0000
Epoch 8/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 57763250044928.0000 - val_loss: 63871956025344.0000
Epoch 9/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 53839730311168.0000 - val_loss: 60361617178624.0000